# Multi-View ResNet 기반 구조물 안정성 예측 - 코드 인수인계 PPT 생성기
이 노트북을 실행하면 코드 구조 분석 및 Flowchart가 포함된 PPT 파일이 생성됩니다.

In [1]:
# ============================================
# 0. Colab Auto-Connect & 패키지 설치
# ============================================
import IPython
from google.colab import output

# Colab 자동 재연결 (세션 끊김 방지)
display(IPython.display.Javascript('''
function KeepClicking(){
  console.log("Colab Auto-Reconnect Active");
  document.querySelector("colab-connect-button").click()
}
setInterval(KeepClicking, 60000)
'''))
print("✅ Colab Auto-Connect 활성화 완료 (60초 간격)")

<IPython.core.display.Javascript object>

✅ Colab Auto-Connect 활성화 완료 (60초 간격)


In [2]:
!pip install python-pptx -q
print("✅ python-pptx 설치 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 4.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 17.2 MB/s eta 0:00:00
✅ python-pptx 설치 완료


In [3]:
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE
import copy

# ============================================
# 색상 팔레트 정의
# ============================================
COLORS = {
    'dark_navy':    RGBColor(0x1B, 0x2A, 0x4A),
    'navy':         RGBColor(0x2C, 0x3E, 0x6B),
    'blue':         RGBColor(0x3B, 0x7D, 0xDD),
    'light_blue':   RGBColor(0x5B, 0xA0, 0xF5),
    'sky':          RGBColor(0xD6, 0xEA, 0xF8),
    'white':        RGBColor(0xFF, 0xFF, 0xFF),
    'light_gray':   RGBColor(0xF0, 0xF2, 0xF5),
    'gray':         RGBColor(0x7F, 0x8C, 0x9A),
    'dark_gray':    RGBColor(0x4A, 0x4A, 0x5A),
    'green':        RGBColor(0x27, 0xAE, 0x60),
    'orange':       RGBColor(0xF3, 0x9C, 0x12),
    'red':          RGBColor(0xE7, 0x4C, 0x3C),
    'purple':       RGBColor(0x8E, 0x44, 0xAD),
    'teal':         RGBColor(0x00, 0x96, 0x88),
    'bg_slide':     RGBColor(0xF8, 0xFA, 0xFC),
}

prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

SLIDE_W = Inches(13.333)
SLIDE_H = Inches(7.5)

# ============================================
# 유틸 함수
# ============================================
def set_slide_bg(slide, color):
    bg = slide.background
    fill = bg.fill
    fill.solid()
    fill.fore_color.rgb = color

def add_shape_with_text(slide, left, top, width, height, text, 
                        fill_color=None, font_color=COLORS['white'],
                        font_size=Pt(14), bold=False, shape_type=MSO_SHAPE.ROUNDED_RECTANGLE,
                        alignment=PP_ALIGN.CENTER):
    shape = slide.shapes.add_shape(shape_type, left, top, width, height)
    if fill_color:
        shape.fill.solid()
        shape.fill.fore_color.rgb = fill_color
    shape.line.fill.background()
    tf = shape.text_frame
    tf.word_wrap = True
    tf.auto_size = None
    p = tf.paragraphs[0]
    p.text = text
    p.font.size = font_size
    p.font.color.rgb = font_color
    p.font.bold = bold
    p.alignment = alignment
    tf.paragraphs[0].space_before = Pt(0)
    tf.paragraphs[0].space_after = Pt(0)
    return shape

def add_textbox(slide, left, top, width, height, text, 
                font_size=Pt(14), font_color=COLORS['dark_gray'], 
                bold=False, alignment=PP_ALIGN.LEFT):
    txBox = slide.shapes.add_textbox(left, top, width, height)
    tf = txBox.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.text = text
    p.font.size = font_size
    p.font.color.rgb = font_color
    p.font.bold = bold
    p.alignment = alignment
    return txBox

def add_multi_text(slide, left, top, width, height, lines, font_size=Pt(13), 
                   font_color=COLORS['dark_gray'], line_spacing=Pt(22), bold_first=False):
    """여러 줄 텍스트를 하나의 텍스트박스에 추가"""
    txBox = slide.shapes.add_textbox(left, top, width, height)
    tf = txBox.text_frame
    tf.word_wrap = True
    for i, line in enumerate(lines):
        if i == 0:
            p = tf.paragraphs[0]
        else:
            p = tf.add_paragraph()
        p.text = line
        p.font.size = font_size
        p.font.color.rgb = font_color
        p.space_before = Pt(2)
        p.space_after = Pt(2)
        if bold_first and i == 0:
            p.font.bold = True
    return txBox

def add_arrow(slide, start_left, start_top, end_left, end_top, color=COLORS['gray']):
    """화살표 커넥터 추가"""
    connector = slide.shapes.add_connector(
        1, start_left, start_top, end_left, end_top
    )
    connector.line.color.rgb = color
    connector.line.width = Pt(2)
    # end arrow
    connector.line.dash_style = None
    from pptx.oxml.ns import qn
    line_elem = connector.line._ln
    tail = line_elem.find(qn('a:tailEnd'))
    if tail is None:
        from lxml import etree
        tail = etree.SubElement(line_elem, qn('a:tailEnd'))
    tail.set('type', 'triangle')
    tail.set('w', 'med')
    tail.set('len', 'med')
    return connector

def add_section_header(slide, title_text, subtitle_text=""):
    """슬라이드 상단 헤더 바 추가"""
    bar = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0), Inches(0), SLIDE_W, Inches(1.1))
    bar.fill.solid()
    bar.fill.fore_color.rgb = COLORS['dark_navy']
    bar.line.fill.background()
    add_textbox(slide, Inches(0.7), Inches(0.15), Inches(10), Inches(0.5),
                title_text, font_size=Pt(28), font_color=COLORS['white'], bold=True)
    if subtitle_text:
        add_textbox(slide, Inches(0.7), Inches(0.6), Inches(10), Inches(0.4),
                    subtitle_text, font_size=Pt(14), font_color=COLORS['sky'])
    # accent line
    accent = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0), Inches(1.1), SLIDE_W, Inches(0.04))
    accent.fill.solid()
    accent.fill.fore_color.rgb = COLORS['blue']
    accent.line.fill.background()

print("✅ 유틸 함수 정의 완료")

✅ 유틸 함수 정의 완료


In [4]:
# ============================================
# 슬라이드 1: 표지
# ============================================
slide = prs.slides.add_slide(prs.slide_layouts[6])  # blank
set_slide_bg(slide, COLORS['dark_navy'])

# 상단 장식 라인
deco = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0), Inches(0), SLIDE_W, Inches(0.06))
deco.fill.solid()
deco.fill.fore_color.rgb = COLORS['blue']
deco.line.fill.background()

# 타이틀
add_textbox(slide, Inches(1), Inches(1.5), Inches(11), Inches(1),
            "Multi-View ResNet 기반", font_size=Pt(22), font_color=COLORS['light_blue'], bold=False, alignment=PP_ALIGN.CENTER)
add_textbox(slide, Inches(1), Inches(2.2), Inches(11), Inches(1.2),
            "구조물 안정성 예측 모델", font_size=Pt(44), font_color=COLORS['white'], bold=True, alignment=PP_ALIGN.CENTER)

# 구분선
line = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(5), Inches(3.5), Inches(3.333), Inches(0.03))
line.fill.solid()
line.fill.fore_color.rgb = COLORS['blue']
line.line.fill.background()

# 부제
add_textbox(slide, Inches(1), Inches(3.8), Inches(11), Inches(0.6),
            "코드 구조 분석 및 인수인계 문서", font_size=Pt(20), font_color=COLORS['sky'], alignment=PP_ALIGN.CENTER)
add_textbox(slide, Inches(1), Inches(4.5), Inches(11), Inches(0.5),
            "Baseline Code Walkthrough & Architecture Flowchart", font_size=Pt(14), font_color=COLORS['gray'], alignment=PP_ALIGN.CENTER)

# 하단 정보
footer = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0), Inches(6.8), SLIDE_W, Inches(0.7))
footer.fill.solid()
footer.fill.fore_color.rgb = COLORS['navy']
footer.line.fill.background()
add_textbox(slide, Inches(1), Inches(6.85), Inches(11), Inches(0.5),
            "Framework: PyTorch  |  Model: ResNet18 (Multi-View)  |  Task: Binary Classification (Stable / Unstable)",
            font_size=Pt(12), font_color=COLORS['light_blue'], alignment=PP_ALIGN.CENTER)

print("✅ 슬라이드 1 (표지) 생성 완료")

✅ 슬라이드 1 (표지) 생성 완료


In [5]:
# ============================================
# 슬라이드 2: 목차
# ============================================
slide = prs.slides.add_slide(prs.slide_layouts[6])
set_slide_bg(slide, COLORS['bg_slide'])
add_section_header(slide, "목차 (Table of Contents)")

toc_items = [
    ("01", "프로젝트 개요", "문제 정의, 데이터 구조, 평가 지표"),
    ("02", "전체 파이프라인 Flowchart", "End-to-End 처리 흐름도"),
    ("03", "환경 설정 & 하이퍼파라미터", "라이브러리, Seed, CFG 설정"),
    ("04", "데이터 로드 & 전처리", "CSV 로드, Dataset 클래스, DataLoader"),
    ("05", "모델 아키텍처 (Multi-View ResNet)", "ResNet18 backbone, Feature Fusion, Classifier"),
    ("06", "학습 & 검증 루프", "BCEWithLogitsLoss, Adam, LogLoss 평가"),
    ("07", "추론 & 제출 파일 생성", "Sigmoid 확률 변환, submission.csv 저장"),
    ("08", "핵심 요약 & 개선 포인트", "인수인계 체크리스트, 개선 방향"),
]

start_y = Inches(1.5)
for i, (num, title, desc) in enumerate(toc_items):
    y = start_y + Inches(i * 0.7)
    # 번호 원
    add_shape_with_text(slide, Inches(1.2), y, Inches(0.55), Inches(0.55), num,
                        fill_color=COLORS['blue'], font_color=COLORS['white'],
                        font_size=Pt(14), bold=True, shape_type=MSO_SHAPE.OVAL)
    # 제목
    add_textbox(slide, Inches(2.0), y - Inches(0.02), Inches(4), Inches(0.35),
                title, font_size=Pt(16), font_color=COLORS['dark_navy'], bold=True)
    # 설명
    add_textbox(slide, Inches(2.0), y + Inches(0.3), Inches(8), Inches(0.3),
                desc, font_size=Pt(11), font_color=COLORS['gray'])

print("✅ 슬라이드 2 (목차) 생성 완료")

✅ 슬라이드 2 (목차) 생성 완료


In [6]:
# ============================================
# 슬라이드 3: 프로젝트 개요
# ============================================
slide = prs.slides.add_slide(prs.slide_layouts[6])
set_slide_bg(slide, COLORS['bg_slide'])
add_section_header(slide, "01. 프로젝트 개요", "Problem Definition & Data Overview")

# 왼쪽 박스: 문제 정의
add_shape_with_text(slide, Inches(0.5), Inches(1.5), Inches(6), Inches(0.5), "📋  문제 정의",
                    fill_color=COLORS['navy'], font_size=Pt(16), bold=True)
add_multi_text(slide, Inches(0.7), Inches(2.1), Inches(5.6), Inches(2.5), [
    "• 목표: 구조물 이미지(Front/Top View)를 보고 안정성을 이진 분류",
    "• 라벨: stable (안정) vs unstable (불안정)",
    "• 입력: 각 샘플당 2장의 이미지 (front.png, top.png)",
    "• 출력: unstable_prob, stable_prob (확률값)",
    "• 평가 지표: Log-Loss (Binary Cross-Entropy)",
], font_size=Pt(13))

# 오른쪽 박스: 데이터 구조
add_shape_with_text(slide, Inches(6.9), Inches(1.5), Inches(6), Inches(0.5), "📁  데이터 구조",
                    fill_color=COLORS['teal'], font_size=Pt(16), bold=True)
add_multi_text(slide, Inches(7.1), Inches(2.1), Inches(5.6), Inches(2.5), [
    "📂 train/          ← 학습 이미지 폴더",
    "   └─ {id}/front.png, top.png",
    "📂 dev/            ← 검증 이미지 폴더",
    "   └─ {id}/front.png, top.png",
    "📂 test/           ← 테스트 이미지 폴더",
    "   └─ {id}/front.png, top.png",
    "📄 train.csv       ← id, label (stable/unstable)",
    "📄 dev.csv         ← id, label",
    "📄 sample_submission.csv  ← id",
], font_size=Pt(12))

# 하단: 핵심 포인트
add_shape_with_text(slide, Inches(0.5), Inches(5.3), Inches(12.3), Inches(1.5),
                    "", fill_color=COLORS['sky'], shape_type=MSO_SHAPE.ROUNDED_RECTANGLE)
add_multi_text(slide, Inches(0.8), Inches(5.4), Inches(11.8), Inches(1.4), [
    "💡 핵심 포인트",
    "• Multi-View 접근: 하나의 구조물을 Front(정면)와 Top(상면) 2개 시점에서 촬영한 이미지를 동시에 활용",
    "• 각 시점의 특징을 ResNet18으로 추출 → Concatenate → FC Layer로 최종 분류",
    "• 제출 시 unstable 확률과 stable 확률(= 1 - unstable_prob)을 모두 저장",
], font_size=Pt(13), bold_first=True)

print("✅ 슬라이드 3 (프로젝트 개요) 생성 완료")

✅ 슬라이드 3 (프로젝트 개요) 생성 완료


In [7]:
# ============================================
# 슬라이드 4: 전체 파이프라인 Flowchart
# ============================================
slide = prs.slides.add_slide(prs.slide_layouts[6])
set_slide_bg(slide, COLORS['bg_slide'])
add_section_header(slide, "02. 전체 파이프라인 Flowchart", "End-to-End Processing Pipeline")

# Flowchart 노드 정의
flow_nodes = [
    ("환경 설정\n(Seed, Device, CFG)", COLORS['navy']),
    ("데이터 로드\n(train/dev/test CSV)", COLORS['teal']),
    ("Dataset 생성\n(MultiViewDataset)", COLORS['teal']),
    ("DataLoader 생성\n(Batch=32)", COLORS['teal']),
    ("모델 초기화\n(MultiViewResNet)", COLORS['purple']),
    ("학습 루프\n(3 Epochs)", COLORS['blue']),
    ("검증 평가\n(LogLoss, Acc)", COLORS['orange']),
    ("테스트 추론\n(Sigmoid→Prob)", COLORS['green']),
    ("제출 파일 저장\n(submission.csv)", COLORS['red']),
]

node_w = Inches(1.25)
node_h = Inches(0.85)
gap = Inches(0.15)
total_w = len(flow_nodes) * (node_w + gap) - gap
start_x = (SLIDE_W - total_w) // 2
y_center = Inches(2.2)

for i, (text, color) in enumerate(flow_nodes):
    x = start_x + i * (node_w + gap)
    add_shape_with_text(slide, x, y_center, node_w, node_h, text,
                        fill_color=color, font_size=Pt(9), bold=True)
    # 화살표
    if i < len(flow_nodes) - 1:
        arrow_start_x = x + node_w
        arrow_end_x = x + node_w + gap
        arrow_y = y_center + node_h // 2
        add_arrow(slide, arrow_start_x, arrow_y, arrow_end_x, arrow_y, COLORS['blue'])

# ---- 상세 Flowchart: 모델 내부 ----
add_textbox(slide, Inches(0.5), Inches(3.5), Inches(12), Inches(0.4),
            "▼ 모델 내부 데이터 흐름 (Multi-View ResNet Forward Pass)",
            font_size=Pt(16), font_color=COLORS['dark_navy'], bold=True)

# 모델 내부 플로우
model_nodes = [
    ("Front Image\n[B,3,224,224]", COLORS['teal'], Inches(0.3)),
    ("ResNet18\nBackbone", COLORS['purple'], Inches(2.4)),
    ("Feature\n[B, 512]", COLORS['blue'], Inches(4.3)),
]
model_nodes2 = [
    ("Top Image\n[B,3,224,224]", COLORS['teal'], Inches(0.3)),
    ("ResNet18\nBackbone\n(공유)", COLORS['purple'], Inches(2.4)),
    ("Feature\n[B, 512]", COLORS['blue'], Inches(4.3)),
]
merge_nodes = [
    ("Concat\n[B, 1024]", COLORS['orange'], Inches(6.2)),
    ("FC: 1024→256\nReLU + Dropout", COLORS['navy'], Inches(8.1)),
    ("FC: 256→1\n(Logit)", COLORS['navy'], Inches(10.0)),
    ("Sigmoid\n→ Prob", COLORS['green'], Inches(11.7)),
]

y_top = Inches(4.1)
y_bot = Inches(5.5)
y_merge = Inches(4.8)
n_w = Inches(1.5)
n_h = Inches(0.7)

# Top row (Front)
for i, (text, color, x) in enumerate(model_nodes):
    add_shape_with_text(slide, x, y_top, n_w, n_h, text,
                        fill_color=color, font_size=Pt(10), bold=True)
    if i < len(model_nodes) - 1:
        add_arrow(slide, x + n_w, y_top + n_h // 2,
                  model_nodes[i+1][2], y_top + n_h // 2, COLORS['gray'])

# Bottom row (Top)
for i, (text, color, x) in enumerate(model_nodes2):
    add_shape_with_text(slide, x, y_bot, n_w, n_h, text,
                        fill_color=color, font_size=Pt(10), bold=True)
    if i < len(model_nodes2) - 1:
        add_arrow(slide, x + n_w, y_bot + n_h // 2,
                  model_nodes2[i+1][2], y_bot + n_h // 2, COLORS['gray'])

# Arrows to merge
add_arrow(slide, model_nodes[2][2] + n_w, y_top + n_h // 2,
          merge_nodes[0][2], y_merge + n_h // 2, COLORS['orange'])
add_arrow(slide, model_nodes2[2][2] + n_w, y_bot + n_h // 2,
          merge_nodes[0][2], y_merge + n_h // 2, COLORS['orange'])

# Merge row
for i, (text, color, x) in enumerate(merge_nodes):
    add_shape_with_text(slide, x, y_merge, n_w, n_h, text,
                        fill_color=color, font_size=Pt(10), bold=True)
    if i < len(merge_nodes) - 1:
        add_arrow(slide, x + n_w, y_merge + n_h // 2,
                  merge_nodes[i+1][2], y_merge + n_h // 2, COLORS['gray'])

print("✅ 슬라이드 4 (Flowchart) 생성 완료")

✅ 슬라이드 4 (Flowchart) 생성 완료


In [8]:
# ============================================
# 슬라이드 5: 환경 설정 & 하이퍼파라미터
# ============================================
slide = prs.slides.add_slide(prs.slide_layouts[6])
set_slide_bg(slide, COLORS['bg_slide'])
add_section_header(slide, "03. 환경 설정 & 하이퍼파라미터", "Libraries, Seed Control, Configuration")

# 왼쪽: 라이브러리
add_shape_with_text(slide, Inches(0.5), Inches(1.5), Inches(5.8), Inches(0.45), "📦  사용 라이브러리",
                    fill_color=COLORS['navy'], font_size=Pt(15), bold=True)
add_multi_text(slide, Inches(0.7), Inches(2.1), Inches(5.4), Inches(3.5), [
    "• torch, torchvision     — 딥러닝 프레임워크 (PyTorch)",
    "• torchvision.models     — 사전학습 ResNet18 로드",
    "• torch.utils.data       — Dataset, DataLoader",
    "• PIL (Pillow)           — 이미지 로드 (.png → RGB)",
    "• pandas                 — CSV 파일 읽기/쓰기",
    "• numpy                  — 수치 연산, LogLoss 계산",
    "• sklearn                — train_test_split (미사용 import)",
    "• tqdm                   — 학습 진행률 표시",
], font_size=Pt(12))

# 오른쪽: 하이퍼파라미터
add_shape_with_text(slide, Inches(7), Inches(1.5), Inches(5.8), Inches(0.45), "⚙️  하이퍼파라미터 (CFG)",
                    fill_color=COLORS['blue'], font_size=Pt(15), bold=True)

cfg_items = [
    ("IMG_SIZE", "224", "입력 이미지 크기 (224×224)"),
    ("EPOCHS", "3", "학습 반복 횟수"),
    ("LEARNING_RATE", "1e-3", "Adam 옵티마이저 학습률"),
    ("BATCH_SIZE", "32", "미니배치 크기"),
    ("SEED", "42", "재현성을 위한 랜덤 시드"),
]

table_y = Inches(2.15)
# 테이블 헤더
add_shape_with_text(slide, Inches(7.1), table_y, Inches(1.6), Inches(0.4), "파라미터",
                    fill_color=COLORS['dark_navy'], font_size=Pt(11), bold=True)
add_shape_with_text(slide, Inches(8.7), table_y, Inches(1.0), Inches(0.4), "값",
                    fill_color=COLORS['dark_navy'], font_size=Pt(11), bold=True)
add_shape_with_text(slide, Inches(9.7), table_y, Inches(3.0), Inches(0.4), "설명",
                    fill_color=COLORS['dark_navy'], font_size=Pt(11), bold=True)

for i, (param, val, desc) in enumerate(cfg_items):
    row_y = table_y + Inches(0.4) + Inches(i * 0.4)
    bg = COLORS['white'] if i % 2 == 0 else COLORS['light_gray']
    add_shape_with_text(slide, Inches(7.1), row_y, Inches(1.6), Inches(0.4), param,
                        fill_color=bg, font_color=COLORS['dark_navy'], font_size=Pt(11), bold=True)
    add_shape_with_text(slide, Inches(8.7), row_y, Inches(1.0), Inches(0.4), val,
                        fill_color=bg, font_color=COLORS['blue'], font_size=Pt(11), bold=True)
    add_shape_with_text(slide, Inches(9.7), row_y, Inches(3.0), Inches(0.4), desc,
                        fill_color=bg, font_color=COLORS['dark_gray'], font_size=Pt(10))

# Seed 설명
add_shape_with_text(slide, Inches(0.5), Inches(5.5), Inches(12.3), Inches(1.3),
                    "", fill_color=COLORS['sky'], shape_type=MSO_SHAPE.ROUNDED_RECTANGLE)
add_multi_text(slide, Inches(0.8), Inches(5.6), Inches(11.8), Inches(1.2), [
    "🔒 Seed 고정 (seed_everything 함수)",
    "• np.random.seed(42) → NumPy 난수 고정",
    "• torch.manual_seed(42) → PyTorch CPU 시드 고정",
    "• torch.cuda.manual_seed(42) → PyTorch CUDA 시드 고정",
    "• torch.backends.cudnn.deterministic = True → cuDNN 결정적 동작 보장",
], font_size=Pt(12), bold_first=True)

print("✅ 슬라이드 5 (환경 설정) 생성 완료")

✅ 슬라이드 5 (환경 설정) 생성 완료


In [9]:
# ============================================
# 슬라이드 6: 데이터 로드 & 전처리
# ============================================
slide = prs.slides.add_slide(prs.slide_layouts[6])
set_slide_bg(slide, COLORS['bg_slide'])
add_section_header(slide, "04. 데이터 로드 & 전처리", "CSV Load → MultiViewDataset → DataLoader")

# CSV 로드
add_shape_with_text(slide, Inches(0.5), Inches(1.5), Inches(4), Inches(0.45), "📄  CSV 데이터 로드",
                    fill_color=COLORS['teal'], font_size=Pt(14), bold=True)
add_multi_text(slide, Inches(0.7), Inches(2.1), Inches(3.8), Inches(1.5), [
    "• train.csv → train_df (학습 데이터)",
    "• dev.csv → val_df (검증 데이터)",
    "• sample_submission.csv → test_df",
    "",
    "CSV 컬럼: id, label",
    "label 매핑: stable→0, unstable→1",
], font_size=Pt(12))

# MultiViewDataset 클래스
add_shape_with_text(slide, Inches(4.8), Inches(1.5), Inches(8), Inches(0.45), "🔧  MultiViewDataset 클래스",
                    fill_color=COLORS['purple'], font_size=Pt(14), bold=True)
add_multi_text(slide, Inches(5.0), Inches(2.1), Inches(7.6), Inches(2.8), [
    "__init__(df, root_dir, transform, is_test)",
    "  → DataFrame, 이미지 루트 경로, 변환 함수, 테스트 모드 플래그",
    "",
    "__getitem__(idx):",
    "  1. sample_id = df.iloc[idx]['id'] 로 폴더명 결정",
    "  2. front.png, top.png 2장을 PIL로 로드 → RGB 변환",
    "  3. transform 적용 (Resize → ToTensor → Normalize)",
    "  4-a. is_test=True  → [front_tensor, top_tensor] 반환",
    "  4-b. is_test=False → ([front_tensor, top_tensor], label) 반환",
], font_size=Pt(12))

# Transform
add_shape_with_text(slide, Inches(0.5), Inches(4.2), Inches(6), Inches(0.45), "🖼️  이미지 Transform",
                    fill_color=COLORS['blue'], font_size=Pt(14), bold=True)
add_multi_text(slide, Inches(0.7), Inches(4.8), Inches(5.8), Inches(2), [
    "Train Transform:",
    "  Resize(224,224) → ToTensor → Normalize",
    "  ※ Augmentation 추가 가능 위치 표시됨",
    "",
    "Test Transform:",
    "  Resize(224,224) → ToTensor → Normalize",
    "",
    "Normalize: mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]",
    "  (ImageNet 사전학습 통계치 사용)",
], font_size=Pt(11))

# DataLoader
add_shape_with_text(slide, Inches(7), Inches(4.2), Inches(5.8), Inches(0.45), "📦  DataLoader 설정",
                    fill_color=COLORS['orange'], font_size=Pt(14), bold=True)
add_multi_text(slide, Inches(7.2), Inches(4.8), Inches(5.4), Inches(2), [
    "• train_loader: batch=32, shuffle=True",
    "• val_loader:   batch=32, shuffle=False",
    "• test_loader:  batch=32, shuffle=False",
    "",
    "각 배치 반환 형태:",
    "  Train/Val: ([front_batch, top_batch], labels)",
    "  Test:      [front_batch, top_batch]",
], font_size=Pt(12))

print("✅ 슬라이드 6 (데이터 로드) 생성 완료")

✅ 슬라이드 6 (데이터 로드) 생성 완료


In [10]:
# ============================================
# 슬라이드 7: 모델 아키텍처
# ============================================
slide = prs.slides.add_slide(prs.slide_layouts[6])
set_slide_bg(slide, COLORS['bg_slide'])
add_section_header(slide, "05. 모델 아키텍처 (Multi-View ResNet)", "ResNet18 Backbone + Feature Fusion + Binary Classifier")

# 모델 구조 설명
add_shape_with_text(slide, Inches(0.5), Inches(1.5), Inches(6), Inches(0.45), "🏗️  MultiViewResNet 클래스 구조",
                    fill_color=COLORS['purple'], font_size=Pt(15), bold=True)

add_multi_text(slide, Inches(0.7), Inches(2.1), Inches(5.8), Inches(4.5), [
    "class MultiViewResNet(nn.Module):",
    "",
    "  ① backbone = ResNet18 (ImageNet 사전학습 가중치)",
    "  ② feature_extractor = ResNet18에서 마지막 FC층 제거",
    "     → nn.Sequential(*list(backbone.children())[:-1])",
    "     → 출력: [B, 512, 1, 1] → view → [B, 512]",
    "",
    "  ③ classifier = 분류기",
    "     → Linear(512×2=1024, 256)",
    "     → ReLU()",
    "     → Dropout(0.2)",
    "     → Linear(256, 1)  ← Logit 출력",
], font_size=Pt(12))

# Forward 과정 다이어그램
add_shape_with_text(slide, Inches(6.9), Inches(1.5), Inches(6), Inches(0.45), "🔀  Forward Pass 흐름",
                    fill_color=COLORS['blue'], font_size=Pt(15), bold=True)

# 다이어그램 그리기
fw_y = Inches(2.2)
box_w = Inches(2.3)
box_h = Inches(0.55)
x_col = Inches(7.1)
x_col2 = Inches(10)

# Front path
add_shape_with_text(slide, x_col, fw_y, box_w, box_h, "Front Image [B,3,224,224]",
                    fill_color=COLORS['teal'], font_size=Pt(10), bold=True)
add_arrow(slide, x_col + box_w, fw_y + box_h // 2, x_col2, fw_y + box_h // 2, COLORS['gray'])
add_shape_with_text(slide, x_col2, fw_y, box_w, box_h, "Feature f1 [B, 512]",
                    fill_color=COLORS['blue'], font_size=Pt(10), bold=True)

# Top path
add_shape_with_text(slide, x_col, fw_y + Inches(0.8), box_w, box_h, "Top Image [B,3,224,224]",
                    fill_color=COLORS['teal'], font_size=Pt(10), bold=True)
add_arrow(slide, x_col + box_w, fw_y + Inches(0.8) + box_h // 2,
          x_col2, fw_y + Inches(0.8) + box_h // 2, COLORS['gray'])
add_shape_with_text(slide, x_col2, fw_y + Inches(0.8), box_w, box_h, "Feature f2 [B, 512]",
                    fill_color=COLORS['blue'], font_size=Pt(10), bold=True)

# 중간 라벨
add_textbox(slide, x_col + Inches(0.5), fw_y + Inches(0.55), Inches(2), Inches(0.3),
            "← 같은 ResNet18 공유 →", font_size=Pt(9), font_color=COLORS['gray'], alignment=PP_ALIGN.CENTER)

# Concat
concat_y = fw_y + Inches(1.9)
add_shape_with_text(slide, Inches(8.5), concat_y, Inches(2.5), box_h, "Concat → [B, 1024]",
                    fill_color=COLORS['orange'], font_size=Pt(11), bold=True)

# Classifier
add_shape_with_text(slide, Inches(7.1), concat_y + Inches(0.8), Inches(5.5), box_h,
                    "Linear(1024→256) → ReLU → Dropout(0.2) → Linear(256→1)",
                    fill_color=COLORS['navy'], font_size=Pt(11), bold=True)

# Output
add_shape_with_text(slide, Inches(8.5), concat_y + Inches(1.6), Inches(2.5), box_h,
                    "Logit → Sigmoid → Prob",
                    fill_color=COLORS['green'], font_size=Pt(11), bold=True)

# 핵심 포인트
add_shape_with_text(slide, Inches(0.5), Inches(6.1), Inches(12.3), Inches(0.8),
                    "", fill_color=COLORS['sky'], shape_type=MSO_SHAPE.ROUNDED_RECTANGLE)
add_multi_text(slide, Inches(0.8), Inches(6.15), Inches(11.8), Inches(0.7), [
    "🔑 Key Point: 두 시점(Front/Top)의 이미지가 동일한 ResNet18 백본을 공유하여 특징 추출 → Concatenate 후 Fully Connected Layer로 분류",
    "     출력은 Logit(raw score)이며, 학습 시 BCEWithLogitsLoss 사용, 추론 시 Sigmoid를 직접 적용하여 확률 변환",
], font_size=Pt(11), bold_first=True)

print("✅ 슬라이드 7 (모델 아키텍처) 생성 완료")

✅ 슬라이드 7 (모델 아키텍처) 생성 완료


In [11]:
# ============================================
# 슬라이드 8: 학습 & 검증 루프
# ============================================
slide = prs.slides.add_slide(prs.slide_layouts[6])
set_slide_bg(slide, COLORS['bg_slide'])
add_section_header(slide, "06. 학습 & 검증 루프", "Training Loop, Validation, Loss & Metrics")

# 학습 루프
add_shape_with_text(slide, Inches(0.5), Inches(1.5), Inches(6), Inches(0.45), "🔄  train_one_epoch() 함수",
                    fill_color=COLORS['blue'], font_size=Pt(15), bold=True)

train_steps = [
    ("1", "model.train()", "학습 모드 활성화"),
    ("2", "views → device로 이동", "GPU/CPU 전송"),
    ("3", "optimizer.zero_grad()", "기울기 초기화"),
    ("4", "outputs = model(views)", "Forward Pass"),
    ("5", "loss = BCEWithLogitsLoss", "손실 계산"),
    ("6", "loss.backward()", "역전파"),
    ("7", "optimizer.step()", "가중치 업데이트"),
]

for i, (num, step, desc) in enumerate(train_steps):
    y = Inches(2.15) + Inches(i * 0.42)
    add_shape_with_text(slide, Inches(0.6), y, Inches(0.35), Inches(0.35), num,
                        fill_color=COLORS['blue'], font_size=Pt(10), bold=True, shape_type=MSO_SHAPE.OVAL)
    add_textbox(slide, Inches(1.05), y, Inches(2.5), Inches(0.35),
                step, font_size=Pt(11), font_color=COLORS['dark_navy'], bold=True)
    add_textbox(slide, Inches(3.5), y, Inches(3), Inches(0.35),
                desc, font_size=Pt(11), font_color=COLORS['gray'])

# 검증 루프
add_shape_with_text(slide, Inches(6.9), Inches(1.5), Inches(6), Inches(0.45), "📊  validate() 함수",
                    fill_color=COLORS['green'], font_size=Pt(15), bold=True)
add_multi_text(slide, Inches(7.1), Inches(2.15), Inches(5.6), Inches(3), [
    "① model.eval() + torch.no_grad()",
    "   → 평가 모드 (Dropout 비활성화, 기울기 미계산)",
    "",
    "② outputs → Sigmoid → 확률 변환",
    "   → probs = torch.sigmoid(outputs)",
    "",
    "③ Log-Loss 직접 계산 (대회 공식 지표)",
    "   eps = 1e-15 (안정성 클리핑)",
    "   p = clip(probs, eps, 1-eps)",
    "   LogLoss = -mean(y·log(p) + (1-y)·log(1-p))",
    "",
    "④ Accuracy 계산",
    "   → (probs > 0.5) == labels 비율",
], font_size=Pt(12))

# 메인 루프
add_shape_with_text(slide, Inches(0.5), Inches(5.3), Inches(12.3), Inches(1.6),
                    "", fill_color=COLORS['sky'], shape_type=MSO_SHAPE.ROUNDED_RECTANGLE)
add_multi_text(slide, Inches(0.8), Inches(5.35), Inches(11.8), Inches(1.5), [
    "🔁 메인 학습 루프 (Main Loop)",
    "• 모델: MultiViewResNet() → GPU",
    "• 손실 함수: BCEWithLogitsLoss (Sigmoid 내장 → 수치적으로 안정)",
    "• 옵티마이저: Adam (lr=1e-3)",
    "• 3 Epoch 반복: 매 에폭마다 train_one_epoch() → validate() → Train Loss, Val LogLoss, Val Acc 출력",
], font_size=Pt(12), bold_first=True)

print("✅ 슬라이드 8 (학습 & 검증) 생성 완료")

✅ 슬라이드 8 (학습 & 검증) 생성 완료


In [12]:
# ============================================
# 슬라이드 9: 추론 & 제출 파일 생성
# ============================================
slide = prs.slides.add_slide(prs.slide_layouts[6])
set_slide_bg(slide, COLORS['bg_slide'])
add_section_header(slide, "07. 추론 & 제출 파일 생성", "Inference Pipeline & Submission")

# 추론 과정 Flowchart
add_textbox(slide, Inches(0.5), Inches(1.5), Inches(12), Inches(0.4),
            "추론 파이프라인 흐름도", font_size=Pt(18), font_color=COLORS['dark_navy'], bold=True)

infer_nodes = [
    ("model.eval()\ntorch.no_grad()", COLORS['navy']),
    ("test_loader에서\n배치 로드", COLORS['teal']),
    ("views →\nGPU 이동", COLORS['teal']),
    ("model(views)\n→ Logit", COLORS['purple']),
    ("Sigmoid\n→ 확률값", COLORS['orange']),
    ("all_probs에\n누적 저장", COLORS['blue']),
]

inf_w = Inches(1.6)
inf_h = Inches(0.8)
inf_gap = Inches(0.25)
total_inf_w = len(infer_nodes) * (inf_w + inf_gap) - inf_gap
inf_start_x = (SLIDE_W - total_inf_w) // 2
inf_y = Inches(2.1)

for i, (text, color) in enumerate(infer_nodes):
    x = inf_start_x + i * (inf_w + inf_gap)
    add_shape_with_text(slide, x, inf_y, inf_w, inf_h, text,
                        fill_color=color, font_size=Pt(10), bold=True)
    if i < len(infer_nodes) - 1:
        add_arrow(slide, x + inf_w, inf_y + inf_h // 2,
                  x + inf_w + inf_gap, inf_y + inf_h // 2, COLORS['blue'])

# 제출 파일 설명
add_shape_with_text(slide, Inches(0.5), Inches(3.5), Inches(6), Inches(0.45), "📝  제출 파일 구조 (submission.csv)",
                    fill_color=COLORS['green'], font_size=Pt(15), bold=True)

# 테이블
tbl_y = Inches(4.15)
add_shape_with_text(slide, Inches(0.7), tbl_y, Inches(1.8), Inches(0.4), "컬럼",
                    fill_color=COLORS['dark_navy'], font_size=Pt(12), bold=True)
add_shape_with_text(slide, Inches(2.5), tbl_y, Inches(3.8), Inches(0.4), "설명",
                    fill_color=COLORS['dark_navy'], font_size=Pt(12), bold=True)

tbl_data = [
    ("id", "테스트 샘플 ID (sample_submission.csv에서 로드)"),
    ("unstable_prob", "Sigmoid 출력값 (불안정 확률)"),
    ("stable_prob", "1.0 - unstable_prob (안정 확률)"),
]
for i, (col, desc) in enumerate(tbl_data):
    row_y = tbl_y + Inches(0.4) + Inches(i * 0.4)
    bg = COLORS['white'] if i % 2 == 0 else COLORS['light_gray']
    add_shape_with_text(slide, Inches(0.7), row_y, Inches(1.8), Inches(0.4), col,
                        fill_color=bg, font_color=COLORS['dark_navy'], font_size=Pt(11), bold=True)
    add_shape_with_text(slide, Inches(2.5), row_y, Inches(3.8), Inches(0.4), desc,
                        fill_color=bg, font_color=COLORS['dark_gray'], font_size=Pt(11))

# 코드 요약
add_shape_with_text(slide, Inches(7), Inches(3.5), Inches(5.8), Inches(0.45), "💻  핵심 코드 요약",
                    fill_color=COLORS['navy'], font_size=Pt(15), bold=True)
add_multi_text(slide, Inches(7.2), Inches(4.15), Inches(5.4), Inches(2.5), [
    "model.eval()",
    "with torch.no_grad():",
    "    for views in test_loader:",
    "        outputs = model(views).view(-1)",
    "        probs = torch.sigmoid(outputs)",
    "        all_probs.extend(probs.cpu().numpy())",
    "",
    "submission = pd.DataFrame({",
    "    'id': test_df['id'],",
    "    'unstable_prob': all_probs,",
    "    'stable_prob': 1.0 - all_probs",
    "})",
    "submission.to_csv('submission.csv', ...)",
], font_size=Pt(11))

# 주의사항
add_shape_with_text(slide, Inches(0.5), Inches(6.1), Inches(12.3), Inches(0.8),
                    "", fill_color=COLORS['sky'], shape_type=MSO_SHAPE.ROUNDED_RECTANGLE)
add_multi_text(slide, Inches(0.8), Inches(6.15), Inches(11.8), Inches(0.7), [
    "⚠️ 주의: 인코딩은 UTF-8-sig (BOM 포함), 컬럼 순서는 id → unstable_prob → stable_prob 순서 유지 필수",
    "     제출 시 확률의 합이 1이 되어야 함 (stable_prob = 1.0 - unstable_prob)",
], font_size=Pt(12), bold_first=True)

print("✅ 슬라이드 9 (추론 & 제출) 생성 완료")

✅ 슬라이드 9 (추론 & 제출) 생성 완료


In [13]:
# ============================================
# 슬라이드 10: 핵심 요약 & 개선 포인트
# ============================================
slide = prs.slides.add_slide(prs.slide_layouts[6])
set_slide_bg(slide, COLORS['bg_slide'])
add_section_header(slide, "08. 핵심 요약 & 개선 포인트", "Handover Checklist & Future Improvements")

# 인수인계 체크리스트
add_shape_with_text(slide, Inches(0.5), Inches(1.5), Inches(6), Inches(0.45), "✅  인수인계 체크리스트",
                    fill_color=COLORS['green'], font_size=Pt(15), bold=True)
add_multi_text(slide, Inches(0.7), Inches(2.1), Inches(5.8), Inches(3.5), [
    "☑ 데이터 경로: ./train, ./dev, ./test 폴더 + CSV 파일",
    "☑ 라벨: stable(0) / unstable(1) 이진 분류",
    "☑ 입력: 각 샘플당 front.png + top.png (2-View)",
    "☑ 모델: ResNet18 (pretrained) × 1개 공유 백본",
    "☑ Feature Fusion: Concatenation (512+512=1024)",
    "☑ 분류기: FC(1024→256→1) + ReLU + Dropout",
    "☑ 손실 함수: BCEWithLogitsLoss",
    "☑ 옵티마이저: Adam (lr=1e-3)",
    "☑ 평가 지표: Binary Log-Loss + Accuracy",
    "☑ 출력: submission.csv (id, unstable_prob, stable_prob)",
], font_size=Pt(12))

# 개선 방향
add_shape_with_text(slide, Inches(6.9), Inches(1.5), Inches(6), Inches(0.45), "🚀  개선 가능 포인트",
                    fill_color=COLORS['orange'], font_size=Pt(15), bold=True)
add_multi_text(slide, Inches(7.1), Inches(2.1), Inches(5.6), Inches(3.5), [
    "① Data Augmentation 추가",
    "   → RandomHorizontalFlip, RandomRotation, ColorJitter 등",
    "",
    "② 더 큰 Backbone 사용",
    "   → ResNet34/50, EfficientNet, ViT 등",
    "",
    "③ 학습 스케줄러 추가",
    "   → CosineAnnealingLR, ReduceLROnPlateau",
    "",
    "④ Cross-Validation",
    "   → K-Fold CV로 일반화 성능 향상",
    "",
    "⑤ Feature Fusion 방법 변경",
    "   → Concat 외에 Attention, Addition, Bilinear 등",
    "",
    "⑥ Epoch 수 증가 & Early Stopping 적용",
], font_size=Pt(11))

# 하단 요약
add_shape_with_text(slide, Inches(0.5), Inches(5.8), Inches(12.3), Inches(1.1),
                    "", fill_color=COLORS['sky'], shape_type=MSO_SHAPE.ROUNDED_RECTANGLE)
add_multi_text(slide, Inches(0.8), Inches(5.85), Inches(11.8), Inches(1.0), [
    "📌 코드 실행 순서 요약",
    "1) 환경 설정 (Seed, Device) → 2) CSV 로드 → 3) Dataset/DataLoader 생성 → 4) 모델 초기화",
    "→ 5) 3 Epoch 학습-검증 루프 → 6) 테스트 추론 → 7) submission.csv 저장 후 제출",
], font_size=Pt(13), bold_first=True)

print("✅ 슬라이드 10 (핵심 요약) 생성 완료")

✅ 슬라이드 10 (핵심 요약) 생성 완료


In [14]:
# ============================================
# 슬라이드 11: 코드 구조 요약 (한 눈에 보기)
# ============================================
slide = prs.slides.add_slide(prs.slide_layouts[6])
set_slide_bg(slide, COLORS['bg_slide'])
add_section_header(slide, "코드 구조 전체 요약 (One-Page Overview)", "Cell-by-Cell Code Map")

cell_info = [
    ("Cell 1", "라이브러리 Import & CFG 설정", "torch, torchvision, PIL, pandas, numpy, tqdm\nCFG: IMG_SIZE=224, EPOCHS=3, LR=1e-3, BATCH=32, SEED=42", COLORS['navy']),
    ("Cell 2", "데이터 로드", "train.csv → train_df\ndev.csv → val_df", COLORS['teal']),
    ("Cell 3", "MultiViewDataset 클래스 정의", "__getitem__: front.png + top.png 로드 → Transform 적용\nis_test 플래그로 학습/추론 모드 분리", COLORS['purple']),
    ("Cell 4", "Transform & DataLoader 생성", "Train/Test Transform 정의 (Resize→ToTensor→Normalize)\ntrain_loader, val_loader, test_loader 생성", COLORS['blue']),
    ("Cell 5", "MultiViewResNet 모델 정의", "ResNet18 backbone(공유) → Feature Concat(1024)\n→ FC(1024→256→1) Classifier", COLORS['purple']),
    ("Cell 6", "train_one_epoch & validate 함수", "학습: Forward→Loss→Backward→Step\n검증: Sigmoid→LogLoss 계산 + Accuracy", COLORS['blue']),
    ("Cell 7", "메인 학습 루프 실행", "BCEWithLogitsLoss + Adam(lr=1e-3)\n3 Epoch: Train Loss / Val LogLoss / Val Acc 출력", COLORS['orange']),
    ("Cell 8", "테스트 추론 & 제출 파일 저장", "model.eval() → Sigmoid → all_probs\nsubmission.csv (id, unstable_prob, stable_prob) 저장", COLORS['green']),
]

row_h = Inches(0.68)
start_y = Inches(1.45)

# 헤더
add_shape_with_text(slide, Inches(0.3), start_y, Inches(1.0), Inches(0.4), "Cell",
                    fill_color=COLORS['dark_navy'], font_size=Pt(11), bold=True)
add_shape_with_text(slide, Inches(1.3), start_y, Inches(3.0), Inches(0.4), "역할",
                    fill_color=COLORS['dark_navy'], font_size=Pt(11), bold=True)
add_shape_with_text(slide, Inches(4.3), start_y, Inches(8.7), Inches(0.4), "핵심 내용",
                    fill_color=COLORS['dark_navy'], font_size=Pt(11), bold=True)

for i, (cell, role, content, color) in enumerate(cell_info):
    y = start_y + Inches(0.4) + Inches(i * row_h)
    add_shape_with_text(slide, Inches(0.3), y, Inches(1.0), row_h, cell,
                        fill_color=color, font_size=Pt(10), bold=True)
    bg = COLORS['white'] if i % 2 == 0 else COLORS['light_gray']
    add_shape_with_text(slide, Inches(1.3), y, Inches(3.0), row_h, role,
                        fill_color=bg, font_color=COLORS['dark_navy'], font_size=Pt(11), bold=True)
    # content box
    shape = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(4.3), y, Inches(8.7), row_h)
    shape.fill.solid()
    shape.fill.fore_color.rgb = bg
    shape.line.fill.background()
    tf = shape.text_frame
    tf.word_wrap = True
    for j, line in enumerate(content.split('\n')):
        if j == 0:
            p = tf.paragraphs[0]
        else:
            p = tf.add_paragraph()
        p.text = line
        p.font.size = Pt(10)
        p.font.color.rgb = COLORS['dark_gray']
        p.space_before = Pt(1)
        p.space_after = Pt(1)

print("✅ 슬라이드 11 (코드 구조 전체 요약) 생성 완료")

✅ 슬라이드 11 (코드 구조 전체 요약) 생성 완료


In [15]:
# ============================================
# 슬라이드 12: 마무리 (Thank You)
# ============================================
slide = prs.slides.add_slide(prs.slide_layouts[6])
set_slide_bg(slide, COLORS['dark_navy'])

# 장식
deco = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0), Inches(0), SLIDE_W, Inches(0.06))
deco.fill.solid()
deco.fill.fore_color.rgb = COLORS['blue']
deco.line.fill.background()

add_textbox(slide, Inches(1), Inches(2.5), Inches(11), Inches(1),
            "Thank You", font_size=Pt(48), font_color=COLORS['white'], bold=True, alignment=PP_ALIGN.CENTER)

line = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(5.5), Inches(3.6), Inches(2.333), Inches(0.03))
line.fill.solid()
line.fill.fore_color.rgb = COLORS['blue']
line.line.fill.background()

add_textbox(slide, Inches(1), Inches(4.0), Inches(11), Inches(0.6),
            "Multi-View ResNet 기반 구조물 안정성 예측", font_size=Pt(20), font_color=COLORS['light_blue'],
            alignment=PP_ALIGN.CENTER)
add_textbox(slide, Inches(1), Inches(4.6), Inches(11), Inches(0.5),
            "코드 인수인계 완료", font_size=Pt(16), font_color=COLORS['gray'],
            alignment=PP_ALIGN.CENTER)

print("✅ 슬라이드 12 (마무리) 생성 완료")

✅ 슬라이드 12 (마무리) 생성 완료


In [16]:
# ============================================
# PPT 저장
# ============================================
output_path = 'Multi-View_ResNet_코드_인수인계.pptx'
prs.save(output_path)
print(f"\n🎉 PPT 저장 완료: {output_path}")
print(f"📊 총 슬라이드 수: {len(prs.slides)} 장")
print("\n슬라이드 구성:")
print("  1. 표지")
print("  2. 목차")
print("  3. 프로젝트 개요")
print("  4. 전체 파이프라인 Flowchart")
print("  5. 환경 설정 & 하이퍼파라미터")
print("  6. 데이터 로드 & 전처리")
print("  7. 모델 아키텍처 (Multi-View ResNet)")
print("  8. 학습 & 검증 루프")
print("  9. 추론 & 제출 파일 생성")
print("  10. 핵심 요약 & 개선 포인트")
print("  11. 코드 구조 전체 요약 (One-Page)")
print("  12. Thank You")


🎉 PPT 저장 완료: Multi-View_ResNet_코드_인수인계.pptx
📊 총 슬라이드 수: 12 장

슬라이드 구성:
  1. 표지
  2. 목차
  3. 프로젝트 개요
  4. 전체 파이프라인 Flowchart
  5. 환경 설정 & 하이퍼파라미터
  6. 데이터 로드 & 전처리
  7. 모델 아키텍처 (Multi-View ResNet)
  8. 학습 & 검증 루프
  9. 추론 & 제출 파일 생성
  10. 핵심 요약 & 개선 포인트
  11. 코드 구조 전체 요약 (One-Page)
  12. Thank You


In [18]:
# Colab에서 파일 다운로드
from google.colab import files
files.download(output_path)
print("📥 다운로드 시작!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 다운로드 시작!
